# CV_01 — Des pixels à un CNN

*Conception pédagogique : Mathieu Weill. Usage personnel d’apprentissage ; droits et crédits tiers conservés.*

**Compétition de travail : [catsdogs](https://www.kaggle.com/competitions/dogs-vs-cats-redux-kernels-edition)**

**Question : Quels motifs un réseau convolutif peut-il apprendre ?**

7 heures pédagogiques, hors pauses. Les exemples sont des jalons techniques courts ; l’essentiel du temps est consacré aux exercices, revues et expériences. **Version étudiant.**

**[Repère de compréhension : Comprendre pixels, tenseurs et convolution](../../../course/chapitres/CV_01/README.md)**


## Préparation et limites

Lire README.md, compétitions et règles. Les données officielles sont téléchargées personnellement et placées dans data/competitions/<slug>/. Aucun remplacement silencieux par des données simulées. Poids préentraînés à préparer avant NLP_02/CV_01/CV_02. Les limites de calcul et d’échanges dépendent du concours et du compte.

## Notions du jour

Dogs vs. Cats Redux est une compétition historique (clôture le 2 mars 2017), retenue pour apprendre le transfert sur de vraies photographies. Ce n’est pas un prix actuellement ouvert. Les images train sont nommées cat.N.jpg/dog.N.jpg ; la cible positive est chien. La métrique est la **log-loss binaire**, à minimiser ; la soumission est `id,label`, probabilité de chien.

On adapte un ResNet18 ImageNet : respecter résolution, canaux et normalisation des poids. Remplacer la tête ne suffit pas ; geler un backbone implique aussi de contrôler BatchNorm. L’augmentation ne touche que le train et doit préserver la cible.

Le split est stratifié par fichier. Vérifier doublons/quasi-doublons et provenance avant de le qualifier d’indépendant. Les entraînements courts servent à vérifier le parcours avant les runs GPU plus longs.

In [ ]:
from pathlib import Path
import sys,os,json,time
import numpy as np,pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
HERE=Path.cwd()
ROOT=next((p if (p/'mllab/competition.py').exists() else p/'student') for p in [HERE,*HERE.parents] if (p/'mllab/competition.py').exists() or (p/'student/mllab/competition.py').exists())
sys.path.insert(0,str(ROOT))
from mllab.competition import SPECS,read_competition,protocol,run_competition,binary_scores,get_folder
KEY='catsdogs';SPEC=SPECS[KEY]
QUICK=os.environ.get('MLLAB_QUICK','0')=='1'
LIMIT=int(os.environ.get('MLLAB_LIMIT','1000' if QUICK else '15000'))
IMAGE_LIMIT=int(os.environ.get('MLLAB_IMAGE_LIMIT','24' if QUICK else '240'))
plt.rcParams.update({'figure.figsize':(10,4),'axes.spines.top':False,'axes.spines.right':False})
print('Compétition :',SPEC['slug'],'métrique :',SPEC['metric'],'sens :',SPEC['direction'])
if KEY!='catsdogs':a,test,sample,protocol_record=protocol(KEY)
else:
 from mllab.vision_competition import image_protocol
 image_protocol(limit=IMAGE_LIMIT)
print('Données officielles trouvées. Profil rapide :',QUICK,'; aucun score Kaggle inventé.')

## Auditer les vrais JPEG et la cible

In [ ]:
from mllab.vision_competition import image_protocol
from PIL import Image
tr,va=image_protocol(limit=IMAGE_LIMIT)
print('Sous-échantillon',len(tr),len(va));display(tr[['image_id','target']].head())
fig,axes=plt.subplots(1,4,figsize=(12,3))
for ax,(_,row) in zip(axes,tr.head(4).iterrows()):
 with Image.open(row.path) as image:ax.imshow(image.convert('RGB'));ax.set_title('chien' if row.target else 'chat');ax.axis('off')
plt.show()
print('Étiquette 1 = chien. Soumettre une probabilité ; log-loss à minimiser.')

## Apprendre une tête sur un backbone gelé

In [ ]:
from mllab.vision_competition import run_images
out,scores,history=run_images(limit=IMAGE_LIMIT,epochs=1 if QUICK else 3,mode='head',allow_download=False)
display(scores);display(history)
print('ResNet18 préentraîné, transformations des poids, backbone en eval quand gelé. Pas d’augmentation de validation.')

## Préparer les erreurs à examiner

In [ ]:
pred=pd.read_csv(out/'predictions.csv')
pred['error']=(pred.probability_dog>=.5)!=pred.target
pred['confidence']=np.maximum(pred.probability_dog,1-pred.probability_dog)
display(pred.loc[pred.error].sort_values('confidence',ascending=False).head(8))
print('Auditer images ambiguës, doublons, résolution, source ; ne pas corriger les labels de test à la main.')

## Atelier autonome : produire des preuves

1. Expliquer comment le label est déduit des fichiers train sans utiliser les noms du test comme cible.
2. Comparer un run sans augmentation et un run avec augmentation train seulement, même split et même budget.
3. Créer une galerie d’erreurs et proposer une collecte/annotation utile sans modifier le test.

Pour chaque essai : hypothèse avant calcul, code/configuration, empreinte du split, métrique officielle, durée, budget, conclusion, prochaine expérience. Chaque membre doit pouvoir expliquer son apport.

In [ ]:
HYPOTHESE = 'À écrire avant le prochain calcul'
CONCLUSION = 'À compléter avec une preuve et une limite'
# Ajouter ici une expérience limitée, puis expliquer la décision de la conserver ou non.

## Questions de sortie

- Pourquoi une image retournée n’a-t-elle pas toujours la même cible ?
- Pourquoi augmenter aussi le test change-t-il la question évaluée ?

**Remise :** un run traçable, une conclusion, un artefact ou une soumission valide et une prochaine hypothèse. Aucun score public ni prix garanti.